<a href="https://colab.research.google.com/github/sanju-subash/RESEARCH-MCA/blob/main/Research_FINAL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
"""
PART A  = EXACT BASE-PAPER REPRODUCTION  (text-format ReAct: Thought / Action / Observation)
          Same prompts, few-shot, parser, 350-char snippets, 5 steps, 5 questions, seed 42,
          EM / F1 / completion / search-behaviour metrics, failure traces, 30% stress test,
          CSV exports.  Every output of the base-paper notebook is printed.

PART B  = PROPOSED METHOD (ReAct + Reflexion, Adaptive Query Reformulation on tool failure)
          Built on the SAME text-format ReAct engine, on the SAME 5 questions, compared
          with Standard ReAct under a clean and a 30% broken-search condition.

V2 ADDITIONS (infrastructure only, methodology unchanged):
  * Checkpoint / resume: finished questions are saved to ./checkpoints and skipped on rerun,
    so a rate limit never throws away work.  Just rerun the cell later.
  * Smart rate-limit handling: reads Groq's "try again in ..." time and waits; if the wait is
    longer than MAX_WAIT_SECONDS (daily token cap) it stops cleanly with progress saved.
  * Token counter: prints total tokens used this run.
  * gpt-oss: reasoning_effort is set (default "low") to save tokens; completion budget is
    configurable (reasoning tokens count against it).

V8 (SINGLE CELL): paste this whole file into ONE Colab cell and run it - nothing else is needed.
  * MODEL = "auto": before the experiment starts, it tests the candidate Gemini models (MODEL_CANDIDATES)
    with a 1-token call and picks the first one that still has free DAILY quota left.  The choice is saved
    in ./checkpoints/_model.txt, so every rerun/resume uses the SAME model (all four result groups stay
    comparable).  Set MODEL = "gemini-2.5-flash" (or any name) to force one model.
  * Quota messages are now provider-neutral and show the free-tier limit Google reports.
  * The live demo runs once only (a rerun after a quota stop does not repeat it).

V7: temporary server errors (HTTP 500/502/503/504, connection drops, timeouts) are retried with a
  backoff instead of becoming API_ERROR rows.

V6: PROVIDER switch.  PROVIDER = "gemini" uses the Google Gemini API (free key from
  aistudio.google.com, stored as GEMINI_API_KEY) through its OpenAI-compatible endpoint, so you can
  finish today without the Groq quota.  PROVIDER = "groq" keeps the previous behaviour.
  Gemini is NOT the base paper's model (Qwen); use ONE model for the whole run and say so in the report.

V5 FIX: salvages the non-JSON form  "arguments": SEARCH: "..."  immediately (no token-burning retries)
  and only retries at a higher temperature when nothing usable can be recovered.

V4 FIX: gpt-oss sometimes wraps its reply in a fake tool call such as
  {"name": "assistant", "arguments": {"Thought": "..."}}.  v4 retries at a slightly higher
  temperature and, if it still happens, salvages the Thought/Action text from that failed call.
  (The cleanest option is still the base-paper model qwen/qwen3.8-27b once its daily quota resets.)

V3 FIX: gpt-oss models on Groq try to use a built-in "search/browser" tool and Groq rejects that with
  HTTP 400 (tool_use_failed).  v3 (a) tells the model it has no tools, (b) retries, and (c) as a last
  resort converts the attempted tool call into the SEARCH action the controller would run anyway.
  The real error text is printed once so you can see it.  Failed (API_ERROR) rows are never reused
  from checkpoints.

Run as one Colab cell or `python this_file.py`.  To start completely fresh, delete ./checkpoints.
"""

import os, re, sys, time, json, string, random, subprocess
from collections import Counter

# ------------------------------------------------------------------ install / import
def _ensure_packages():
    needed = {"groq": "groq", "openai": "openai", "tavily": "tavily-python",
              "datasets": "datasets", "pandas": "pandas"}
    missing = []
    for mod, pkg in needed.items():
        try:
            __import__(mod)
        except ImportError:
            missing.append(pkg)
    if missing:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U",
                               "--break-system-packages", *missing])

_ensure_packages()

import pandas as pd
from datasets import load_dataset
from groq import Groq
from groq import RateLimitError as _GroqRateLimit
try:
    from groq import BadRequestError as _GroqBadRequest
except ImportError:                                  # very old SDKs
    class _GroqBadRequest(Exception):
        pass
from openai import OpenAI
from openai import RateLimitError as _OAIRateLimit
from openai import BadRequestError as _OAIBadRequest
from tavily import TavilyClient

# tuples work directly in `except` clauses, so the rest of the code is provider-agnostic
RateLimitError = (_GroqRateLimit, _OAIRateLimit)
BadRequestError = (_GroqBadRequest, _OAIBadRequest)

# ------------------------------------------------------------------ configuration (base paper values)
PROVIDER = "gemini"               # "gemini" (free AI Studio key) or "groq"
MODEL = "auto"                    # "auto" = first Gemini model in MODEL_CANDIDATES that still has daily quota
                                  # (or force one, e.g. "gemini-2.5-flash"; groq: "qwen/qwen3.8-27b", "openai/gpt-oss-120b")
MODEL_CANDIDATES = ["gemini-2.5-flash", "gemini-2.5-flash-lite", "gemini-3.1-flash-lite",
                    "gemini-flash-lite-latest", "gemini-3.5-flash-lite"]
GEMINI_REASONING_EFFORT = "none"  # keeps Gemini close to a plain (non-thinking) model, like the base paper
N_QUESTIONS = 5
MAX_STEPS = 5
SEED = 42
MAX_SEARCH_RESULTS = 3

STRESS_RATE = 0.3            # base paper: 30% of searches return "No results found."
STRESS_SEED = None           # None = unseeded like the base paper; set an int (e.g. 123) for repeatability

RUN_DEMO = True              # Kindle demo (a few thousand tokens)
RUN_PROPOSED = True          # Part B (Reflexion).  Set False to run only the base-paper reproduction.
MAX_REFLEXION_ATTEMPTS = 3
MAX_MEMORY_REFLECTIONS = 3
FAULT_SEED = 2026            # Part B: identical fault schedule for both arms

# infrastructure
INTER_CALL_SLEEP = 5.0       # seconds between LLM calls; Gemini free tier is limited per minute
MAX_COMPLETION_TOKENS = 1500 # base paper used 800 (non-reasoning budget); reasoning models need headroom
REASONING_EFFORT = "low"     # used only for openai/gpt-oss-* models
MAX_API_RETRIES = 6
MAX_WAIT_SECONDS = 900       # if the API asks us to wait longer than this (daily cap), stop cleanly
CKPT_DIR = "checkpoints"

client = None
tv = None
TOKENS_USED = 0


def _secret(name):
    try:
        from google.colab import userdata
        v = userdata.get(name)
        if v:
            return v
    except Exception:
        pass
    v = os.environ.get(name)
    if v:
        return v
    from getpass import getpass
    return getpass(f"Enter {name}: ")


def init_clients():
    global client, tv
    tv = TavilyClient(api_key=_secret("TAVILY_API_KEY"))
    if PROVIDER == "gemini":
        client = OpenAI(api_key=_secret("GEMINI_API_KEY"),
                        base_url="https://generativelanguage.googleapis.com/v1beta/openai/")
        print("Gemini and Tavily connections initialized successfully.")
        models = [m.id.replace("models/", "") for m in client.models.list().data]
        print("Available Gemini models:", [x for x in models if x.startswith("gemini")])
    else:
        client = Groq(api_key=_secret("GROQ_API_KEY"))
        print("Groq and Tavily connections initialized successfully.")
        models = [m.id for m in client.models.list().data]
        print("Available models:")
        print(models)
    if MODEL == "auto":
        if PROVIDER != "gemini":
            raise RuntimeError('MODEL = "auto" works only with PROVIDER = "gemini"; set MODEL explicitly.')
        _choose_model(models)
    if MODEL not in models:
        raise RuntimeError(f"{MODEL} not available to this key. Pick one from the list printed above.")
    print("\nExperiment Configuration")
    print("-------------------------")
    print("Model:", MODEL)
    print("Questions:", N_QUESTIONS)
    print("Maximum Steps:", MAX_STEPS)
    print("Random Seed:", SEED)


def _probe_model(name):
    """1-token test call. Returns (usable, note). Only a used-up DAILY quota counts as unusable."""
    try:
        client.chat.completions.create(model=name, temperature=0, max_tokens=8,
                                       messages=[{"role": "user", "content": "Reply with the word OK."}])
        return True, "OK"
    except RateLimitError as e:
        text = str(e)
        if "perday" in text.lower().replace(" ", "").replace("_", ""):
            lim = re.search(r"limit:\s*(\d+)", text)
            return False, "daily free quota used up" + (f" (limit {lim.group(1)} requests/day)" if lim else "")
        return True, "per-minute limit right now (fine, the run will wait)"
    except Exception as e:
        return False, f"error {type(e).__name__}"


def _choose_model(listed):
    global MODEL
    os.makedirs(CKPT_DIR, exist_ok=True)
    pin = os.path.join(CKPT_DIR, "_model.txt")
    if os.path.exists(pin):
        saved = open(pin, encoding="utf-8").read().strip()
        if saved in listed:
            MODEL = saved
            print(f"Using the model chosen earlier (kept so all groups stay comparable): {MODEL}")
            return
    print("Choosing a Gemini model that still has free daily quota ...")
    for name in MODEL_CANDIDATES:
        if name not in listed:
            print(f"  {name}: not available to this key")
            continue
        ok, note = _probe_model(name)
        print(f"  {name}: {note}")
        if ok:
            MODEL = name
            with open(pin, "w", encoding="utf-8") as f:
                f.write(name)
            return
    raise RuntimeError("Every candidate model has used up its free daily quota. Wait for the reset "
                       "(midnight Pacific time = about 1:30 PM India time) or enable billing on the key.")


# ------------------------------------------------------------------ checkpoint helpers
def _ckpt_path(name):
    os.makedirs(CKPT_DIR, exist_ok=True)
    return os.path.join(CKPT_DIR, f"{name}_{MODEL.replace('/', '_')}_n{N_QUESTIONS}.json")


def _ckpt_load(name, default):
    if not name:
        return default
    p = _ckpt_path(name)
    if os.path.exists(p):
        try:
            with open(p, encoding="utf-8") as f:
                return json.load(f)
        except Exception:
            pass
    return default


def _ckpt_save(name, data):
    if name:
        with open(_ckpt_path(name), "w", encoding="utf-8") as f:
            json.dump(data, f, ensure_ascii=False, default=str)


# ------------------------------------------------------------------ LLM access with smart rate-limit handling
def _retry_after_seconds(message):
    """Parse Groq's 'Please try again in 1h2m3.4s / 2m33.36s / 12.5s' hint. None if absent."""
    m = re.search(r"try again in\s+((?:\d+h)?(?:\d+m)?(?:[\d.]+s)?(?:[\d.]+ms)?)", message or "")
    g1 = re.search(r"retry in\s+([\d.]+)s", message or "", flags=re.I) or \
        re.search(r"retryDelay['\"]?\s*:\s*['\"]?([\d.]+)s", message or "")
    if g1:                                           # Gemini style hints
        return float(g1.group(1))
    if "perday" in (message or "").lower().replace(" ", "").replace("_", ""):
        return 99999.0                               # Gemini daily quota: stop cleanly
    if not m or not m.group(1):
        return None
    t = m.group(1)
    h = re.search(r"(\d+)h", t)
    mi = re.search(r"(\d+)m(?!s)", t)
    s = re.search(r"([\d.]+)s", t)
    ms = re.search(r"([\d.]+)ms", t)
    total = (int(h.group(1)) * 3600 if h else 0) + (int(mi.group(1)) * 60 if mi else 0) \
        + (float(s.group(1)) if s else 0) + (float(ms.group(1)) / 1000 if ms else 0)
    return total


_BAD_REQUEST_SHOWN = False
NO_TOOLS_NOTE = ("\n\nIMPORTANT: You have NO tools. Do not call any function, tool, browser or search "
                 "API. Reply with plain text only, using exactly one 'Thought:' line and one 'Action:' line. "
                 "The controller runs the search for you when you write Action: SEARCH.")


def _with_no_tools_note(messages):
    msgs = [dict(m) for m in messages]
    if msgs and msgs[0]["role"] == "system" and NO_TOOLS_NOTE not in msgs[0]["content"]:
        msgs[0]["content"] += NO_TOOLS_NOTE
    return msgs


def _query_from_failed_generation(text):
    m = (re.search(r'query\\?"\s*:\s*\\?"(.+?)\\?"', text)
         or re.search(r"query['\"]?\s*:\s*['\"](.+?)['\"]", text))
    return m.group(1).strip() if m else None


def _salvage_from_failed_generation(err, require_action=False):
    """Turn a rejected pseudo-tool-call into ReAct text.  Returns None if nothing usable."""
    raw = None
    body = getattr(err, "body", None)
    if isinstance(body, dict):
        inner = body.get("error", body)
        if isinstance(inner, dict):
            raw = inner.get("failed_generation")
    text = str(err)
    if raw is None:
        m = re.search(r"failed_generation['\"]?\s*:\s*['\"](.*)['\"]\s*\}", text, flags=re.S)
        raw = m.group(1) if m else text
    args = {}
    obj = None
    if isinstance(raw, str):
        for cand in (raw, raw.replace('\\"', '"')):      # raw JSON first, then un-escaped variant
            try:
                obj = json.loads(cand)
                break
            except Exception:
                continue
    elif isinstance(raw, dict):
        obj = raw
    if isinstance(obj, dict):
        a = obj.get("arguments", obj)
        args = a if isinstance(a, dict) else {}
    if not args:                                 # not valid JSON, e.g.  "arguments": SEARCH: "query text"}
        for src in (raw if isinstance(raw, str) else "", text):
            mm = re.search(r'\b(SEARCH|FINISH)\s*:\s*\\*"(.+?)\\*"\s*\}', src)
            if mm:
                return f'Thought: Continuing.\nAction: {mm.group(1)}: "{mm.group(2).strip()}"'
    lowered = {str(k).lower(): v for k, v in args.items()}
    thought = str(lowered.get("thought", "")).strip()
    action = str(lowered.get("action", "")).strip()
    query = lowered.get("query") or lowered.get("q")
    if action:
        return f"Thought: {thought or 'Continuing.'}\nAction: {action}"
    q = (str(query).strip() if query else None) or _query_from_failed_generation(text)
    if q:
        return f'Thought: {thought or "I need to look this up."}\nAction: SEARCH: "{q}"'
    if thought and not require_action:
        # No action: return only the thought -> the controller counts an invalid action and the
        # model gets another step to produce a proper Action.
        return f"Thought: {thought}"
    return None


def _call_llm(messages, max_tokens=None):
    global TOKENS_USED
    max_tokens = max_tokens or MAX_COMPLETION_TOKENS
    use_effort = (PROVIDER == "gemini") or MODEL.startswith("openai/gpt-oss")
    tool_fails = 0
    temperature = 0
    for attempt in range(MAX_API_RETRIES + 3):
        kwargs = dict(model=MODEL, messages=messages, temperature=temperature)
        kwargs["max_tokens" if PROVIDER == "gemini" else "max_completion_tokens"] = max_tokens
        if use_effort:
            kwargs["reasoning_effort"] = GEMINI_REASONING_EFFORT if PROVIDER == "gemini" else REASONING_EFFORT
        try:
            response = client.chat.completions.create(**kwargs)
            try:
                TOKENS_USED += response.usage.total_tokens
            except Exception:
                pass
            if INTER_CALL_SLEEP:
                time.sleep(INTER_CALL_SLEEP)
            return response.choices[0].message.content or ""
        except RateLimitError as e:
            hint = _retry_after_seconds(str(e))
            wait = (hint + 2) if hint is not None else 15 * (attempt + 1)
            if wait > MAX_WAIT_SECONDS or attempt == MAX_API_RETRIES - 1:
                print(f"Rate limit: the API asks for ~{wait/60:.1f} min of waiting (daily quota of {MODEL} is "
                      "probably used up). Stopping cleanly; progress is saved - rerun later to resume.")
                raise
            print(f"Rate limit reached. Retry {attempt + 1}/{MAX_API_RETRIES - 1}. "
                  f"Waiting {wait:.0f} seconds...")
            time.sleep(wait)
        except BadRequestError as e:
            text = str(e)
            global _BAD_REQUEST_SHOWN
            if not _BAD_REQUEST_SHOWN:
                _BAD_REQUEST_SHOWN = True
                print("  [BadRequestError details] " + text[:600])
            low = text.lower()
            if use_effort and "tool" not in low and "failed_generation" not in low:
                use_effort = False          # model rejected reasoning_effort; retry once without it
                continue
            if "tool" in low or "failed_generation" in low:
                tool_fails += 1
                messages = _with_no_tools_note(messages)
                salvaged = _salvage_from_failed_generation(e, require_action=True)
                if salvaged:                # an Action/query was recoverable: use it, no extra tokens
                    return salvaged
                if tool_fails <= 2:         # temp=0 repeats the same failure; vary slightly
                    temperature = 0.6
                    continue
                salvaged = _salvage_from_failed_generation(e)
                if salvaged:                # thought only: counted as an invalid action, model retries
                    return salvaged
            raise
        except Exception as e:                  # temporary server / network problems: retry with backoff
            code = getattr(e, "status_code", None)
            transient = (code in (500, 502, 503, 504) or
                         type(e).__name__ in ("InternalServerError", "APIConnectionError",
                                              "APITimeoutError", "ServiceUnavailableError"))
            if transient and attempt < MAX_API_RETRIES + 2:
                wait = min(90, 15 * (attempt + 1))
                print(f"Temporary server error ({type(e).__name__}). Retrying in {wait} seconds...")
                time.sleep(wait)
                continue
            raise


# =====================================================================================
#                                 PART A : BASE PAPER
# =====================================================================================
SYSTEM_PROMPT = """
You are an intelligent question-answering agent
following the ReAct framework.

For every response, generate exactly:
1. One Thought line.
2. One Action line.

Use ONLY these formats:

Thought: <brief reasoning>

Action: SEARCH: "<search query>"

OR

Thought: <brief reasoning>

Action: FINISH: "<final answer>"

Rules:
1. Generate exactly one Thought and one Action per response.
2. SEARCH when you need external information.
3. FINISH when you have sufficient evidence to answer.
4. Never generate an Observation. The search tool provides it.
5. Do not include multiple alternative actions.
6. Do not use Markdown, bold text, code blocks, or headings.
7. Do not add conversational filler.
8. Keep the Thought concise.
9. Put the Action on a separate line.
10. Do not invent facts. Base your answer on the question
    and the available observations.
"""

FEW_SHOT = """
Example:

Question: What country was the scientist who discovered
penicillin born in?

Thought 1: I need to identify the scientist who discovered
penicillin first.

Action 1: SEARCH: "Who discovered penicillin?"

Observation 1: Penicillin was discovered by Alexander Fleming.

Thought 2: Now I need to identify Alexander Fleming's
birthplace.

Action 2: SEARCH: "Alexander Fleming birthplace"

Observation 2: Alexander Fleming was born in Lochfield,
Scotland.

Thought 3: The scientist was born in Scotland.

Action 3: FINISH: "Scotland"

Now solve the actual question.
"""


def chat(prompt):
    messages = [{"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": prompt}]
    if PROVIDER == "groq" and MODEL.startswith("openai/gpt-oss"):   # prevents native tool-call attempts
        messages = _with_no_tools_note(messages)
    return _call_llm(messages)


def web_search(query):
    try:
        response = tv.search(query=query, max_results=MAX_SEARCH_RESULTS)
        results = response.get("results", [])
        if not results:
            return "No results found.", True, False
        observations = []
        for result in results:
            title = result.get("title", "Untitled")
            content = result.get("content", "")
            url = result.get("url", "")
            observations.append(f"Title: {title}\nContent: {content[:350]}\nSource: {url}")
        return "\n\n".join(observations), False, False
    except Exception as error:
        return f"Search error: {type(error).__name__}: {error}", False, True


_original_web_search = web_search


def parse_react_output(output):
    cleaned = output.strip()
    cleaned = cleaned.replace("**", "")
    cleaned = cleaned.replace("__", "")
    cleaned = cleaned.replace("```", "")
    cleaned = cleaned.replace("`", "")

    action_match = re.search(
        r"(?im)^\s*(?:Action\s*\d*\s*:\s*)?"
        r"(SEARCH|FINISH)\s*:\s*(.*?)\s*$",
        cleaned
    )
    if not action_match:
        return None, None

    action_type = action_match.group(1).upper()
    action_input = action_match.group(2).strip()

    if (len(action_input) >= 2 and action_input[0] == action_input[-1]
            and action_input[0] in ("'", '"')):
        action_input = action_input[1:-1].strip()

    before_action = cleaned[:action_match.start()].strip()
    thought_match = re.search(r"(?is)Thought\s*\d*\s*:\s*(.*)", before_action)
    thought = thought_match.group(1).strip() if thought_match else before_action
    return action_type, (action_input, thought)


def react(question, max_steps=None, verbose=False, reflections=""):
    """Base-paper ReAct loop. `reflections` (default empty) is used only by Part B."""
    if max_steps is None:
        max_steps = MAX_STEPS

    trace = ""
    queries = []
    empty_searches = 0
    search_errors = 0
    invalid_actions = 0
    answer = None
    status = "step_limit"
    steps_used = 0

    reflection_block = ""
    if reflections and reflections.strip():
        reflection_block = ("\nReflections from earlier failed attempts "
                            "(use them to choose better, differently-worded search queries "
                            "and do not repeat failed queries):\n" + reflections + "\n")

    for step in range(1, max_steps + 1):
        steps_used = step
        prompt = (FEW_SHOT + reflection_block + "\nQuestion: " + question + "\n" + trace
                  + f"\nThought {step}:")

        output = chat(prompt).strip()
        action_type, parsed = parse_react_output(output)

        if action_type is None:
            invalid_actions += 1
            observation = ("Invalid action format. Return exactly one Thought and one Action. "
                           "Use SEARCH: or FINISH:.")
            trace += (f"\nThought {step}: Unable to parse model response.\n"
                      f"Action {step}: INVALID\n"
                      f"Observation {step}: {observation}\n")
            if verbose:
                print("\nRAW INVALID MODEL OUTPUT:")
                print(repr(output))
            continue

        action_input, thought = parsed

        if action_type == "FINISH":
            if not action_input:
                invalid_actions += 1
                trace += (f"\nThought {step}: {thought}\n"
                          f"Action {step}: INVALID\n"
                          f"Observation {step}: Empty final answer.\n")
                continue
            answer = action_input
            status = "finished"
            trace += f"\nThought {step}: {thought}\n" + f'Action {step}: FINISH: "{answer}"\n'
            break

        if action_type == "SEARCH":
            if not action_input:
                invalid_actions += 1
                trace += (f"\nThought {step}: {thought}\n"
                          f"Action {step}: INVALID\n"
                          f"Observation {step}: Empty search query.\n")
                continue
            query = action_input
            queries.append(query)
            observation, is_empty, is_error = web_search(query)   # global lookup (stress test swaps it)
            if is_empty:
                empty_searches += 1
            if is_error:
                search_errors += 1
            trace += (f"\nThought {step}: {thought}\n"
                      f'Action {step}: SEARCH: "{query}"\n'
                      f"Observation {step}: {observation}\n")

    normalized_queries = [" ".join(q.lower().split()) for q in queries]
    repeated_query = len(normalized_queries) != len(set(normalized_queries))

    if verbose:
        print("=" * 80)
        print("QUESTION:")
        print(question)
        print("=" * 80)
        print(trace)
        print("-" * 80)
        print("FINAL ANSWER:", answer)
        print("STATUS:", status)
        print("INVALID ACTIONS:", invalid_actions)
        print("=" * 80)

    return {
        "question": question, "answer": answer, "status": status, "steps": steps_used,
        "n_search": len(queries), "queries": queries, "empty_searches": empty_searches,
        "search_errors": search_errors, "invalid_actions": invalid_actions,
        "repeated_query": repeated_query, "trace": trace
    }


# ---------------------------------------------------------------- evaluation metrics (base paper)
def normalize_answer(text):
    text = (text or "").lower()
    text = "".join(char for char in text if char not in string.punctuation)
    text = re.sub(r"\b(a|an|the)\b", " ", text)
    return " ".join(text.split())


def exact_match(prediction, gold):
    return int(normalize_answer(prediction) == normalize_answer(gold))


def token_f1(prediction, gold):
    prediction_tokens = normalize_answer(prediction).split()
    gold_tokens = normalize_answer(gold).split()
    if not prediction_tokens or not gold_tokens:
        return float(prediction_tokens == gold_tokens)
    common = Counter(prediction_tokens) & Counter(gold_tokens)
    common_count = sum(common.values())
    if common_count == 0:
        return 0.0
    precision = common_count / len(prediction_tokens)
    recall = common_count / len(gold_tokens)
    return 2 * precision * recall / (precision + recall)


def score_result(result, gold):
    em = exact_match(result["answer"], gold)
    f1 = token_f1(result["answer"], gold)
    if result["status"] == "api_error":
        outcome = "API_ERROR"
    elif result["status"] != "finished":
        outcome = "NO_ANSWER (step limit)"
    elif em == 1:
        outcome = "EXACT_MATCH"
    elif f1 >= 0.5:
        outcome = "PARTIAL_MATCH"
    else:
        outcome = "WRONG"
    return em, f1, outcome


def _api_error_result(question, error):
    return {"question": question, "answer": None, "status": "api_error", "steps": 0, "n_search": 0,
            "queries": [], "empty_searches": 0, "search_errors": 0, "invalid_actions": 0,
            "repeated_query": False, "trace": f"API error: {error}",
            "attempts": 1, "first_attempt_failed": False, "recovered": False}


def _drop_failed_tail(results):
    for i, r in enumerate(results):
        if r.get("status") == "api_error":
            return results[:i]
    return results


def _valid_ckpt(results, rows):
    return all(i < len(rows) and r.get("question") == rows[i]["question"] for i, r in enumerate(results))


def run_batch(rows, ckpt_name=None):
    results = _drop_failed_tail(_ckpt_load(ckpt_name, []))
    if results and not _valid_ckpt(results, rows):
        print("Checkpoint does not match current sample; starting fresh.")
        results = []
    if results:
        print(f"Resuming from checkpoint: {len(results)}/{len(rows)} questions already done.")

    for i, row in enumerate(rows):
        if i < len(results):
            continue
        print(f"\nRunning Question {i+1}/{len(rows)}")
        try:
            result = react(row["question"])
        except RateLimitError:
            print("\nExperiment stopped due to the API rate limit. Partial results kept; rerun to resume.")
            break
        except Exception as error:
            print(f"API error on question {i+1}: {type(error).__name__}")
            result = _api_error_result(row["question"], error)

        result["gold"] = row["answer"]
        result["em"], result["f1"], result["outcome"] = score_result(result, row["answer"])
        results.append(result)
        _ckpt_save(ckpt_name, results)
        print(f"Q{i+1:02d} | {result['outcome']} | gold={row['answer']!r} | "
              f"got={result['answer']!r} | searches={result['n_search']}")
    return pd.DataFrame(results)


def show_summary(df, title="REACT BASELINE EVALUATION RESULTS"):
    if len(df) == 0:
        print("No questions completed (rate limit?). Nothing to summarise.")
        return
    total = len(df)
    print("=" * 55)
    print(title)
    print("=" * 55)
    print("Total Questions:", total)
    print("Exact Match:", f"{df['em'].mean():.1%}")
    print("Average Token F1:", f"{df['f1'].mean():.3f}")
    print("Completion Rate:", f"{(df['status'] == 'finished').mean():.1%}")
    print("Questions with Empty Search:", f"{(df['empty_searches'] > 0).mean():.1%}")
    print("Questions with Search Errors:", f"{(df['search_errors'] > 0).mean():.1%}")
    print("Repeated Query Frequency:", f"{df['repeated_query'].mean():.1%}")
    print("Average Searches per Question:", f"{df['n_search'].mean():.2f}")
    print("\nOutcome Distribution:")
    print(df["outcome"].value_counts().to_string())
    print("=" * 55)


def show_failures(df, k=3):
    if len(df) == 0:
        print("No results to show.")
        return
    failures = df[df["outcome"].isin(["WRONG", "NO_ANSWER (step limit)"])]
    print("Representative unsuccessful examples")
    for idx, row in failures.head(k).iterrows():
        print("\n" + "=" * 80)
        print(f"QUESTION {idx + 1}:")
        print(row["question"])
        print("\nGOLD ANSWER:")
        print(row["gold"])
        print("\nPREDICTED ANSWER:")
        print(row["answer"] if isinstance(row["answer"], str) else "(no answer)")
        print("\nOUTCOME:")
        print(row["outcome"])
        print("\nREACT TRACE:")
        print(row["trace"])


def load_sample():
    dataset = load_dataset("hotpotqa/hotpot_qa", "distractor", split="validation")
    dataset = dataset.select_columns(["question", "answer", "type", "level"])
    rows = [{"question": it["question"], "answer": it["answer"], "type": it["type"], "level": it["level"]}
            for it in dataset]
    hard_questions = [r for r in rows if r["type"] == "bridge" and r["level"] == "hard"]
    random.Random(SEED).shuffle(hard_questions)
    sample = hard_questions[:N_QUESTIONS]
    print("Total validation questions:", len(rows))
    print("Hard bridge questions:", len(hard_questions))
    print("Selected questions:", len(sample))
    print("\nSample question:")
    print(sample[0]["question"])
    return sample


def run_stress_test(sample):
    """Base-paper Section C: 30% of searches replaced by 'No results found.'"""
    global web_search
    print(f"Running Stress Test with {int(STRESS_RATE * 100)}% broken searches...\n")
    rng = random.Random(STRESS_SEED) if STRESS_SEED is not None else random

    def broken_web_search(query):
        if rng.random() < STRESS_RATE:
            return "No results found.", True, False
        return _original_web_search(query)

    web_search = broken_web_search
    try:
        df_stress = run_batch(sample, "stress")
    finally:
        web_search = _original_web_search
    if len(df_stress) == len(sample):
        print("\n--- STRESS TEST SUMMARY ---")
        show_summary(df_stress)
        print("\n--- STRESS TEST FAILURES ---")
        show_failures(df_stress, k=3)
    return df_stress


# =====================================================================================
#                 PART B : PROPOSED METHOD  (ReAct + Reflexion)
# =====================================================================================
def llm_call(system, user, max_tokens=800):
    return _call_llm([{"role": "system", "content": system}, {"role": "user", "content": user}],
                     max_tokens=max_tokens)


def generate_reflection(question, res, reason):
    prompt = (
        "A search-based question answering attempt failed. Do not answer the question.\n\n"
        f"Question: {question}\n"
        f"Search queries used: {res['queries']}\n"
        f"Failure reason: {reason}\n"
        f"Attempt trace (end):\n{res['trace'][-2500:]}\n\n"
        "Write exactly three short lines:\n"
        "Failure cause: <why it failed>\n"
        "Avoid: <strategy not to repeat>\n"
        "Next search query: <a better, differently-worded plain search query>\n"
        "No Markdown, no extra text.")
    text = llm_call("You are a reflection generator. You have no tools.", prompt)
    text = re.sub(r"<think>.*?</think>", "", text, flags=re.S).strip()
    return " | ".join(line.strip() for line in text.splitlines() if line.strip())


def attempt_ok(res):
    """Label-free operational evaluator: answer produced AND at least one non-empty observation."""
    evidence = res["n_search"] - res["empty_searches"] - res["search_errors"]
    if res["status"] != "finished" or not (res["answer"] or "").strip():
        return False, (f"No final answer within the step limit (empty searches={res['empty_searches']}, "
                       f"search errors={res['search_errors']}, invalid actions={res['invalid_actions']}).")
    if evidence <= 0:
        return False, "Answer produced without any non-empty search evidence."
    return True, "ok"


def react_reflexion(question, max_attempts=None):
    max_attempts = max_attempts or MAX_REFLEXION_ATTEMPTS
    memory, attempts = [], []
    for a in range(1, max_attempts + 1):
        refl = "\n".join(f"- {m}" for m in memory[-MAX_MEMORY_REFLECTIONS:])
        res = react(question, reflections=refl)
        ok, reason = attempt_ok(res)
        res["_ok"], res["_reason"], res["_refl_supplied"] = ok, reason, bool(refl)
        attempts.append(res)
        if ok:
            break
        if a < max_attempts:
            try:
                memory.append(generate_reflection(question, res, reason))
            except RateLimitError:
                raise
            except Exception as e:
                memory.append(f"(reflection failed: {type(e).__name__})")

    final = attempts[-1]
    queries = [q for r in attempts for q in r["queries"]]
    norm = [" ".join(q.lower().split()) for q in queries]
    first_failed = not attempts[0]["_ok"]
    return {
        "question": question,
        "answer": final["answer"] if final["_ok"] else None,
        "status": "finished" if final["_ok"] else final["status"],
        "steps": sum(r["steps"] for r in attempts),
        "n_search": len(queries), "queries": queries,
        "empty_searches": sum(r["empty_searches"] for r in attempts),
        "search_errors": sum(r["search_errors"] for r in attempts),
        "invalid_actions": sum(r["invalid_actions"] for r in attempts),
        "repeated_query": len(norm) != len(set(norm)),
        "attempts": len(attempts),
        "first_attempt_failed": first_failed,
        "recovered": bool(first_failed and final["_ok"]),
        "reflections": memory,
        "trace": "\n\n".join(f"##### ATTEMPT {i+1} ({'PASS' if r['_ok'] else 'FAIL: ' + r['_reason']}) #####\n"
                             + r["trace"] for i, r in enumerate(attempts)),
    }


def _faulty_search(rate, key):
    state = {"n": 0}

    def fn(query):
        state["n"] += 1
        if rate > 0 and random.Random(f"{key}-{state['n']}").random() < rate:
            return "No results found.", True, False
        return _original_web_search(query)
    return fn


def run_pair_batch(sample, rate, label):
    """Standard ReAct vs ReAct+Reflexion on the same questions with the SAME fault schedule.
    Returns (df_std, df_ref, complete)."""
    global web_search
    name = f"pair_{label}"
    ck = _ckpt_load(name, {"std": [], "ref": []})
    std_rows, ref_rows = ck["std"], ck["ref"]
    if len(std_rows) != len(ref_rows) or not _valid_ckpt(std_rows, sample):
        std_rows, ref_rows = [], []
    for i in range(len(std_rows)):
        if std_rows[i].get("status") == "api_error" or ref_rows[i].get("status") == "api_error":
            std_rows, ref_rows = std_rows[:i], ref_rows[:i]
            break
    if std_rows:
        print(f"[{label}] Resuming from checkpoint: {len(std_rows)}/{len(sample)} questions done.")

    complete = True
    for i, row in enumerate(sample):
        if i < len(std_rows):
            continue
        key = f"{FAULT_SEED}-{label}-{i}"
        print(f"\n[{label}] Question {i+1}/{len(sample)}")
        pair = []
        try:
            for arm, fn in (("Standard ", react), ("Reflexion", react_reflexion)):
                web_search = _faulty_search(rate, key)
                try:
                    res = fn(row["question"])
                except RateLimitError:
                    raise
                except Exception as e:
                    res = _api_error_result(row["question"], e)
                finally:
                    web_search = _original_web_search
                res["gold"] = row["answer"]
                res["em"], res["f1"], res["outcome"] = score_result(res, row["answer"])
                pair.append(res)
                extra = f" attempts={res.get('attempts', 1)}" if arm == "Reflexion" else ""
                print(f"  {arm}: {res['outcome']:<24} got={res['answer']!r} gold={row['answer']!r}{extra}")
        except RateLimitError:
            print("  Rate limit: stopping; progress saved. Rerun later to resume.")
            complete = False
            break
        std_rows.append(pair[0])
        ref_rows.append(pair[1])
        _ckpt_save(name, {"std": std_rows, "ref": ref_rows})
    return pd.DataFrame(std_rows), pd.DataFrame(ref_rows), complete


def print_pair_report(df_std, df_ref, label):
    show_summary(df_std, f"[{label}] STANDARD REACT (BASE PAPER)")
    show_summary(df_ref, f"[{label}] REACT + REFLEXION (PROPOSED)")
    if len(df_ref) == 0 or "first_attempt_failed" not in df_ref.columns:
        return
    ff = int(df_ref["first_attempt_failed"].sum())
    print(f"Reflexion: first attempt failed on {ff}/{len(df_ref)} questions; "
          f"recovered after reflection on {int(df_ref['recovered'].sum())}; "
          f"avg attempts = {df_ref['attempts'].mean():.2f}")
    wins = int((df_ref["em"] > df_std["em"]).sum())
    losses = int((df_ref["em"] < df_std["em"]).sum())
    print(f"Paired (EM): Reflexion-only correct={wins}, Standard-only correct={losses}, "
          f"delta EM={df_ref['em'].mean() - df_std['em'].mean():+.3f}, "
          f"delta F1={df_ref['f1'].mean() - df_std['f1'].mean():+.3f}")


# =====================================================================================
#                                      MAIN
# =====================================================================================
BASE_PAPER_REFERENCE = {
    "clean  (5 q)": {"EM": "20.0%", "F1": "0.467", "Completion": "60.0%", "Avg searches": "3.20",
                     "Empty/Errors/Repeated": "0% / 0% / 0%"},
    "stress (5 q)": {"EM": "20.0%", "F1": "0.333", "Completion": "40.0%", "Avg searches": "4.20",
                     "Empty/Errors/Repeated": "60% / 20% / 20%"},
}


def _incomplete(stage):
    print(f"\n*** {stage} is incomplete (rate limit). Progress is saved in ./{CKPT_DIR}. "
          f"Rerun this cell later (or after the quota resets) to continue. ***")
    print(f"Tokens used this run: {TOKENS_USED:,}")


def main():
    init_clients()

    demo_flag = os.path.join(CKPT_DIR, "_demo_done.txt")
    if RUN_DEMO and not os.path.exists(demo_flag):
        print("\n\n######## LIVE DEMONSTRATION (not part of benchmark) ########")
        demo_question = "Which university did the founder of the company that makes the Kindle attend?"
        try:
            react(demo_question, verbose=True)
        except RateLimitError:
            _incomplete("Demo")
            return
        with open(demo_flag, "w", encoding="utf-8") as f:
            f.write("done")

    print("\n\n######## HOTPOTQA DATASET ########")
    sample = load_sample()

    print("\n\n######## BASELINE RUN ########")
    df_baseline = run_batch(sample, "baseline")
    if len(df_baseline) < len(sample):
        _incomplete("Baseline run")
        return
    if (df_baseline["status"] == "api_error").any():
        print("\n*** Some questions ended in API_ERROR (see the error line printed above). "
              "Fix the cause and rerun; failed rows are not reused from checkpoints. ***")
        return
    show_summary(df_baseline)
    show_failures(df_baseline, k=3)

    print("\n\n######## SECTION C: 30% STRESS TEST ########")
    df_stress = run_stress_test(sample)
    if len(df_stress) < len(sample):
        _incomplete("Stress test")
        return

    df_baseline.to_csv("react_baseline_results.csv", index=False)
    pd.DataFrame(sample).to_csv("react_baseline_sample.csv", index=False)
    df_stress.to_csv("react_stress_results.csv", index=False)
    print("Experiment files saved successfully.")

    print("\n\n######## BASE-PAPER REPORTED VALUES (for comparison with the run above) ########")
    print(pd.DataFrame(BASE_PAPER_REFERENCE).to_string())

    if RUN_PROPOSED:
        print("\n\n######## PART B: PROPOSED METHOD (ReAct + Reflexion) ########")
        out = {}
        for label, rate in (("clean", 0.0), ("stress_30pct", STRESS_RATE)):
            d_std, d_ref, complete = run_pair_batch(sample, rate, label)
            print_pair_report(d_std, d_ref, label)
            d_std.to_csv(f"proposed_{label}_standard.csv", index=False)
            d_ref.to_csv(f"proposed_{label}_reflexion.csv", index=False)
            out[label] = (d_std, d_ref)
            if not complete:
                _incomplete(f"Part B ({label})")
                break
        if "stress_30pct" in out and len(out["stress_30pct"][1]) == len(sample):
            print("\nFailure traces (Reflexion, stress condition):")
            show_failures(out["stress_30pct"][1], k=2)

    print(f"\nTotal tokens used this run: {TOKENS_USED:,}")

    try:
        from google.colab import files
        for f in os.listdir("."):
            if f.endswith(".csv") and (f.startswith("react_") or f.startswith("proposed_")):
                files.download(f)
    except Exception:
        pass


if __name__ == "__main__":
    main()

Gemini and Tavily connections initialized successfully.
Available Gemini models: ['gemini-2.5-flash', 'gemini-2.5-pro', 'gemini-2.5-flash-preview-tts', 'gemini-2.5-pro-preview-tts', 'gemini-flash-latest', 'gemini-flash-lite-latest', 'gemini-pro-latest', 'gemini-2.5-flash-lite', 'gemini-2.5-flash-image', 'gemini-3-flash-preview', 'gemini-3.1-pro-preview', 'gemini-3.1-pro-preview-customtools', 'gemini-3.1-flash-lite-preview', 'gemini-3.1-flash-lite', 'gemini-3-pro-image-preview', 'gemini-3-pro-image', 'gemini-3.1-flash-image-preview', 'gemini-3.1-flash-image', 'gemini-3.1-flash-lite-image', 'gemini-nano-banana-2.1', 'gemini-3.5-flash', 'gemini-3.5-flash-lite', 'gemini-omni-flash-preview', 'gemini-omni-1.1-flash', 'gemini-omni-1.2-flash', 'gemini-3.5-transcribe', 'gemini-3.6-flash', 'gemini-3.7-flash', 'gemini-3.8-flash', 'gemini-4-argon', 'gemini-3.1-flash-tts-preview', 'gemini-3.8-flash-tts', 'gemini-3.8-flash-lite-tts', 'gemini-robotics-er-2-preview', 'gemini-2.5-computer-use-preview-1

In [5]:
"""
PART A  = EXACT BASE-PAPER REPRODUCTION  (text-format ReAct: Thought / Action / Observation)
          Same prompts, few-shot, parser, 350-char snippets, 5 steps, 5 questions, seed 42,
          EM / F1 / completion / search-behaviour metrics, failure traces, 30% stress test,
          CSV exports.  Every output of the base-paper notebook is printed.

PART B  = PROPOSED METHOD (ReAct + Reflexion, Adaptive Query Reformulation on tool failure)
          Built on the SAME text-format ReAct engine, on the SAME 5 questions, compared
          with Standard ReAct under a clean and a 30% broken-search condition.

V2 ADDITIONS (infrastructure only, methodology unchanged):
  * Checkpoint / resume: finished questions are saved to ./checkpoints and skipped on rerun,
    so a rate limit never throws away work.  Just rerun the cell later.
  * Smart rate-limit handling: reads Groq's "try again in ..." time and waits; if the wait is
    longer than MAX_WAIT_SECONDS (daily token cap) it stops cleanly with progress saved.
  * Token counter: prints total tokens used this run.
  * gpt-oss: reasoning_effort is set (default "low") to save tokens; completion budget is
    configurable (reasoning tokens count against it).

V8 (SINGLE CELL): paste this whole file into ONE Colab cell and run it - nothing else is needed.
  * MODEL = "auto": before the experiment starts, it tests the candidate Gemini models (MODEL_CANDIDATES)
    with a 1-token call and picks the first one that still has free DAILY quota left.  The choice is saved
    in ./checkpoints/_model.txt, so every rerun/resume uses the SAME model (all four result groups stay
    comparable).  Set MODEL = "gemini-2.5-flash" (or any name) to force one model.
  * Quota messages are now provider-neutral and show the free-tier limit Google reports.
  * The live demo runs once only (a rerun after a quota stop does not repeat it).

V7: temporary server errors (HTTP 500/502/503/504, connection drops, timeouts) are retried with a
  backoff instead of becoming API_ERROR rows.

V6: PROVIDER switch.  PROVIDER = "gemini" uses the Google Gemini API (free key from
  aistudio.google.com, stored as GEMINI_API_KEY) through its OpenAI-compatible endpoint, so you can
  finish today without the Groq quota.  PROVIDER = "groq" keeps the previous behaviour.
  Gemini is NOT the base paper's model (Qwen); use ONE model for the whole run and say so in the report.

V5 FIX: salvages the non-JSON form  "arguments": SEARCH: "..."  immediately (no token-burning retries)
  and only retries at a higher temperature when nothing usable can be recovered.

V4 FIX: gpt-oss sometimes wraps its reply in a fake tool call such as
  {"name": "assistant", "arguments": {"Thought": "..."}}.  v4 retries at a slightly higher
  temperature and, if it still happens, salvages the Thought/Action text from that failed call.
  (The cleanest option is still the base-paper model qwen/qwen3.8-27b once its daily quota resets.)

V3 FIX: gpt-oss models on Groq try to use a built-in "search/browser" tool and Groq rejects that with
  HTTP 400 (tool_use_failed).  v3 (a) tells the model it has no tools, (b) retries, and (c) as a last
  resort converts the attempted tool call into the SEARCH action the controller would run anyway.
  The real error text is printed once so you can see it.  Failed (API_ERROR) rows are never reused
  from checkpoints.

Run as one Colab cell or `python this_file.py`.  To start completely fresh, delete ./checkpoints.
"""

import os, re, sys, time, json, string, random, subprocess
from collections import Counter

# ------------------------------------------------------------------ install / import
def _ensure_packages():
    needed = {"groq": "groq", "openai": "openai", "tavily": "tavily-python",
              "datasets": "datasets", "pandas": "pandas"}
    missing = []
    for mod, pkg in needed.items():
        try:
            __import__(mod)
        except ImportError:
            missing.append(pkg)
    if missing:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U",
                               "--break-system-packages", *missing])

_ensure_packages()

import pandas as pd
from datasets import load_dataset
from groq import Groq
from groq import RateLimitError as _GroqRateLimit
try:
    from groq import BadRequestError as _GroqBadRequest
except ImportError:                                  # very old SDKs
    class _GroqBadRequest(Exception):
        pass
from openai import OpenAI
from openai import RateLimitError as _OAIRateLimit
from openai import BadRequestError as _OAIBadRequest
from tavily import TavilyClient

# tuples work directly in `except` clauses, so the rest of the code is provider-agnostic
RateLimitError = (_GroqRateLimit, _OAIRateLimit)
BadRequestError = (_GroqBadRequest, _OAIBadRequest)

# ------------------------------------------------------------------ configuration (base paper values)
PROVIDER = "gemini"               # "gemini" (free AI Studio key) or "groq"
MODEL = "gemini-3.5-flash-lite"                    # "auto" = first Gemini model in MODEL_CANDIDATES that still has daily quota
                                  # (or force one, e.g. "gemini-2.5-flash"; groq: "qwen/qwen3.8-27b", "openai/gpt-oss-120b")
MODEL_CANDIDATES = ["gemini-3.5-flash-lite", "gemini-3.1-flash-lite", "gemini-flash-lite-latest",
                    "gemini-2.5-flash"]   # gemini-2.5-flash-lite is closed to new keys; 2.5-flash is out of quota today
GEMINI_REASONING_EFFORT = "none"  # keeps Gemini close to a plain (non-thinking) model, like the base paper
N_QUESTIONS = 5
MAX_STEPS = 5
SEED = 42
MAX_SEARCH_RESULTS = 3

STRESS_RATE = 0.3            # base paper: 30% of searches return "No results found."
STRESS_SEED = None           # None = unseeded like the base paper; set an int (e.g. 123) for repeatability

RUN_DEMO = True              # Kindle demo (a few thousand tokens)
RUN_PROPOSED = True          # Part B (Reflexion).  Set False to run only the base-paper reproduction.
MAX_REFLEXION_ATTEMPTS = 3
MAX_MEMORY_REFLECTIONS = 3
FAULT_SEED = 2026            # Part B: identical fault schedule for both arms

# infrastructure
INTER_CALL_SLEEP = 5.0       # seconds between LLM calls; Gemini free tier is limited per minute
MAX_COMPLETION_TOKENS = 1500 # base paper used 800 (non-reasoning budget); reasoning models need headroom
REASONING_EFFORT = "low"     # used only for openai/gpt-oss-* models
MAX_API_RETRIES = 6
MAX_WAIT_SECONDS = 900       # if the API asks us to wait longer than this (daily cap), stop cleanly
CKPT_DIR = "checkpoints"

client = None
tv = None
TOKENS_USED = 0


def _secret(name):
    try:
        from google.colab import userdata
        v = userdata.get(name)
        if v:
            return v
    except Exception:
        pass
    v = os.environ.get(name)
    if v:
        return v
    from getpass import getpass
    return getpass(f"Enter {name}: ")


def init_clients():
    global client, tv
    tv = TavilyClient(api_key=_secret("TAVILY_API_KEY"))
    if PROVIDER == "gemini":
        client = OpenAI(api_key=_secret("GEMINI_API_KEY"),
                        base_url="https://generativelanguage.googleapis.com/v1beta/openai/")
        print("Gemini and Tavily connections initialized successfully.")
        models = [m.id.replace("models/", "") for m in client.models.list().data]
        print("Available Gemini models:", [x for x in models if x.startswith("gemini")])
    else:
        client = Groq(api_key=_secret("GROQ_API_KEY"))
        print("Groq and Tavily connections initialized successfully.")
        models = [m.id for m in client.models.list().data]
        print("Available models:")
        print(models)
    if MODEL == "auto":
        if PROVIDER != "gemini":
            raise RuntimeError('MODEL = "auto" works only with PROVIDER = "gemini"; set MODEL explicitly.')
        _choose_model(models)
    if MODEL not in models:
        raise RuntimeError(f"{MODEL} not available to this key. Pick one from the list printed above.")
    print("\nExperiment Configuration")
    print("-------------------------")
    print("Model:", MODEL)
    print("Questions:", N_QUESTIONS)
    print("Maximum Steps:", MAX_STEPS)
    print("Random Seed:", SEED)


def _probe_model(name):
    """1-token test call. Returns (usable, note). Only a used-up DAILY quota counts as unusable."""
    try:
        client.chat.completions.create(model=name, temperature=0, max_tokens=8,
                                       messages=[{"role": "user", "content": "Reply with the word OK."}])
        return True, "OK"
    except RateLimitError as e:
        text = str(e)
        if "perday" in text.lower().replace(" ", "").replace("_", ""):
            lim = re.search(r"limit:\s*(\d+)", text)
            return False, "daily free quota used up" + (f" (limit {lim.group(1)} requests/day)" if lim else "")
        return True, "per-minute limit right now (fine, the run will wait)"
    except Exception as e:
        return False, f"error {type(e).__name__}"


def _choose_model(listed):
    global MODEL
    os.makedirs(CKPT_DIR, exist_ok=True)
    pin = os.path.join(CKPT_DIR, "_model.txt")
    if os.path.exists(pin) and os.path.exists(os.path.join(CKPT_DIR, "_reset_pin.txt")):
        os.remove(pin)
    if os.path.exists(pin):
        saved = open(pin, encoding="utf-8").read().strip()
        if saved in listed:
            MODEL = saved
            print(f"Using the model chosen earlier (kept so all groups stay comparable): {MODEL}")
            return
    print("Choosing a Gemini model that still has free daily quota ...")
    for name in MODEL_CANDIDATES:
        if name not in listed:
            print(f"  {name}: not available to this key")
            continue
        ok, note = _probe_model(name)
        print(f"  {name}: {note}")
        if ok:
            MODEL = name
            with open(pin, "w", encoding="utf-8") as f:
                f.write(name)
            return
    raise RuntimeError("Every candidate model has used up its free daily quota. Wait for the reset "
                       "(midnight Pacific time = about 1:30 PM India time) or enable billing on the key.")


# ------------------------------------------------------------------ checkpoint helpers
def _ckpt_path(name):
    os.makedirs(CKPT_DIR, exist_ok=True)
    return os.path.join(CKPT_DIR, f"{name}_{MODEL.replace('/', '_')}_n{N_QUESTIONS}.json")


def _ckpt_load(name, default):
    if not name:
        return default
    p = _ckpt_path(name)
    if os.path.exists(p):
        try:
            with open(p, encoding="utf-8") as f:
                return json.load(f)
        except Exception:
            pass
    return default


def _ckpt_save(name, data):
    if name:
        with open(_ckpt_path(name), "w", encoding="utf-8") as f:
            json.dump(data, f, ensure_ascii=False, default=str)


# ------------------------------------------------------------------ LLM access with smart rate-limit handling
def _retry_after_seconds(message):
    """Parse Groq's 'Please try again in 1h2m3.4s / 2m33.36s / 12.5s' hint. None if absent."""
    m = re.search(r"try again in\s+((?:\d+h)?(?:\d+m)?(?:[\d.]+s)?(?:[\d.]+ms)?)", message or "")
    g1 = re.search(r"retry in\s+([\d.]+)s", message or "", flags=re.I) or \
        re.search(r"retryDelay['\"]?\s*:\s*['\"]?([\d.]+)s", message or "")
    if g1:                                           # Gemini style hints
        return float(g1.group(1))
    if "perday" in (message or "").lower().replace(" ", "").replace("_", ""):
        return 99999.0                               # Gemini daily quota: stop cleanly
    if not m or not m.group(1):
        return None
    t = m.group(1)
    h = re.search(r"(\d+)h", t)
    mi = re.search(r"(\d+)m(?!s)", t)
    s = re.search(r"([\d.]+)s", t)
    ms = re.search(r"([\d.]+)ms", t)
    total = (int(h.group(1)) * 3600 if h else 0) + (int(mi.group(1)) * 60 if mi else 0) \
        + (float(s.group(1)) if s else 0) + (float(ms.group(1)) / 1000 if ms else 0)
    return total


_BAD_REQUEST_SHOWN = False
NO_TOOLS_NOTE = ("\n\nIMPORTANT: You have NO tools. Do not call any function, tool, browser or search "
                 "API. Reply with plain text only, using exactly one 'Thought:' line and one 'Action:' line. "
                 "The controller runs the search for you when you write Action: SEARCH.")


def _with_no_tools_note(messages):
    msgs = [dict(m) for m in messages]
    if msgs and msgs[0]["role"] == "system" and NO_TOOLS_NOTE not in msgs[0]["content"]:
        msgs[0]["content"] += NO_TOOLS_NOTE
    return msgs


def _query_from_failed_generation(text):
    m = (re.search(r'query\\?"\s*:\s*\\?"(.+?)\\?"', text)
         or re.search(r"query['\"]?\s*:\s*['\"](.+?)['\"]", text))
    return m.group(1).strip() if m else None


def _salvage_from_failed_generation(err, require_action=False):
    """Turn a rejected pseudo-tool-call into ReAct text.  Returns None if nothing usable."""
    raw = None
    body = getattr(err, "body", None)
    if isinstance(body, dict):
        inner = body.get("error", body)
        if isinstance(inner, dict):
            raw = inner.get("failed_generation")
    text = str(err)
    if raw is None:
        m = re.search(r"failed_generation['\"]?\s*:\s*['\"](.*)['\"]\s*\}", text, flags=re.S)
        raw = m.group(1) if m else text
    args = {}
    obj = None
    if isinstance(raw, str):
        for cand in (raw, raw.replace('\\"', '"')):      # raw JSON first, then un-escaped variant
            try:
                obj = json.loads(cand)
                break
            except Exception:
                continue
    elif isinstance(raw, dict):
        obj = raw
    if isinstance(obj, dict):
        a = obj.get("arguments", obj)
        args = a if isinstance(a, dict) else {}
    if not args:                                 # not valid JSON, e.g.  "arguments": SEARCH: "query text"}
        for src in (raw if isinstance(raw, str) else "", text):
            mm = re.search(r'\b(SEARCH|FINISH)\s*:\s*\\*"(.+?)\\*"\s*\}', src)
            if mm:
                return f'Thought: Continuing.\nAction: {mm.group(1)}: "{mm.group(2).strip()}"'
    lowered = {str(k).lower(): v for k, v in args.items()}
    thought = str(lowered.get("thought", "")).strip()
    action = str(lowered.get("action", "")).strip()
    query = lowered.get("query") or lowered.get("q")
    if action:
        return f"Thought: {thought or 'Continuing.'}\nAction: {action}"
    q = (str(query).strip() if query else None) or _query_from_failed_generation(text)
    if q:
        return f'Thought: {thought or "I need to look this up."}\nAction: SEARCH: "{q}"'
    if thought and not require_action:
        # No action: return only the thought -> the controller counts an invalid action and the
        # model gets another step to produce a proper Action.
        return f"Thought: {thought}"
    return None


def _call_llm(messages, max_tokens=None):
    global TOKENS_USED
    max_tokens = max_tokens or MAX_COMPLETION_TOKENS
    use_effort = (PROVIDER == "gemini") or MODEL.startswith("openai/gpt-oss")
    tool_fails = 0
    temperature = 0
    for attempt in range(MAX_API_RETRIES + 3):
        kwargs = dict(model=MODEL, messages=messages, temperature=temperature)
        kwargs["max_tokens" if PROVIDER == "gemini" else "max_completion_tokens"] = max_tokens
        if use_effort:
            kwargs["reasoning_effort"] = GEMINI_REASONING_EFFORT if PROVIDER == "gemini" else REASONING_EFFORT
        try:
            response = client.chat.completions.create(**kwargs)
            try:
                TOKENS_USED += response.usage.total_tokens
            except Exception:
                pass
            if INTER_CALL_SLEEP:
                time.sleep(INTER_CALL_SLEEP)
            return response.choices[0].message.content or ""
        except RateLimitError as e:
            hint = _retry_after_seconds(str(e))
            wait = (hint + 2) if hint is not None else 15 * (attempt + 1)
            if wait > MAX_WAIT_SECONDS or attempt == MAX_API_RETRIES - 1:
                print(f"Rate limit: the API asks for ~{wait/60:.1f} min of waiting (daily quota of {MODEL} is "
                      "probably used up). Stopping cleanly; progress is saved - rerun later to resume.")
                raise
            print(f"Rate limit reached. Retry {attempt + 1}/{MAX_API_RETRIES - 1}. "
                  f"Waiting {wait:.0f} seconds...")
            time.sleep(wait)
        except BadRequestError as e:
            text = str(e)
            global _BAD_REQUEST_SHOWN
            if not _BAD_REQUEST_SHOWN:
                _BAD_REQUEST_SHOWN = True
                print("  [BadRequestError details] " + text[:600])
            low = text.lower()
            if use_effort and "tool" not in low and "failed_generation" not in low:
                use_effort = False          # model rejected reasoning_effort; retry once without it
                continue
            if "tool" in low or "failed_generation" in low:
                tool_fails += 1
                messages = _with_no_tools_note(messages)
                salvaged = _salvage_from_failed_generation(e, require_action=True)
                if salvaged:                # an Action/query was recoverable: use it, no extra tokens
                    return salvaged
                if tool_fails <= 2:         # temp=0 repeats the same failure; vary slightly
                    temperature = 0.6
                    continue
                salvaged = _salvage_from_failed_generation(e)
                if salvaged:                # thought only: counted as an invalid action, model retries
                    return salvaged
            raise
        except Exception as e:                  # temporary server / network problems: retry with backoff
            code = getattr(e, "status_code", None)
            transient = (code in (500, 502, 503, 504) or
                         type(e).__name__ in ("InternalServerError", "APIConnectionError",
                                              "APITimeoutError", "ServiceUnavailableError"))
            if transient and attempt < MAX_API_RETRIES + 2:
                wait = min(90, 15 * (attempt + 1))
                print(f"Temporary server error ({type(e).__name__}). Retrying in {wait} seconds...")
                time.sleep(wait)
                continue
            raise


# =====================================================================================
#                                 PART A : BASE PAPER
# =====================================================================================
SYSTEM_PROMPT = """
You are an intelligent question-answering agent
following the ReAct framework.

For every response, generate exactly:
1. One Thought line.
2. One Action line.

Use ONLY these formats:

Thought: <brief reasoning>

Action: SEARCH: "<search query>"

OR

Thought: <brief reasoning>

Action: FINISH: "<final answer>"

Rules:
1. Generate exactly one Thought and one Action per response.
2. SEARCH when you need external information.
3. FINISH when you have sufficient evidence to answer.
4. Never generate an Observation. The search tool provides it.
5. Do not include multiple alternative actions.
6. Do not use Markdown, bold text, code blocks, or headings.
7. Do not add conversational filler.
8. Keep the Thought concise.
9. Put the Action on a separate line.
10. Do not invent facts. Base your answer on the question
    and the available observations.
"""

FEW_SHOT = """
Example:

Question: What country was the scientist who discovered
penicillin born in?

Thought 1: I need to identify the scientist who discovered
penicillin first.

Action 1: SEARCH: "Who discovered penicillin?"

Observation 1: Penicillin was discovered by Alexander Fleming.

Thought 2: Now I need to identify Alexander Fleming's
birthplace.

Action 2: SEARCH: "Alexander Fleming birthplace"

Observation 2: Alexander Fleming was born in Lochfield,
Scotland.

Thought 3: The scientist was born in Scotland.

Action 3: FINISH: "Scotland"

Now solve the actual question.
"""


def chat(prompt):
    messages = [{"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": prompt}]
    if PROVIDER == "groq" and MODEL.startswith("openai/gpt-oss"):   # prevents native tool-call attempts
        messages = _with_no_tools_note(messages)
    return _call_llm(messages)


def web_search(query):
    try:
        response = tv.search(query=query, max_results=MAX_SEARCH_RESULTS)
        results = response.get("results", [])
        if not results:
            return "No results found.", True, False
        observations = []
        for result in results:
            title = result.get("title", "Untitled")
            content = result.get("content", "")
            url = result.get("url", "")
            observations.append(f"Title: {title}\nContent: {content[:350]}\nSource: {url}")
        return "\n\n".join(observations), False, False
    except Exception as error:
        return f"Search error: {type(error).__name__}: {error}", False, True


_original_web_search = web_search


def parse_react_output(output):
    cleaned = output.strip()
    cleaned = cleaned.replace("**", "")
    cleaned = cleaned.replace("__", "")
    cleaned = cleaned.replace("```", "")
    cleaned = cleaned.replace("`", "")

    action_match = re.search(
        r"(?im)^\s*(?:Action\s*\d*\s*:\s*)?"
        r"(SEARCH|FINISH)\s*:\s*(.*?)\s*$",
        cleaned
    )
    if not action_match:
        return None, None

    action_type = action_match.group(1).upper()
    action_input = action_match.group(2).strip()

    if (len(action_input) >= 2 and action_input[0] == action_input[-1]
            and action_input[0] in ("'", '"')):
        action_input = action_input[1:-1].strip()

    before_action = cleaned[:action_match.start()].strip()
    thought_match = re.search(r"(?is)Thought\s*\d*\s*:\s*(.*)", before_action)
    thought = thought_match.group(1).strip() if thought_match else before_action
    return action_type, (action_input, thought)


def react(question, max_steps=None, verbose=False, reflections=""):
    """Base-paper ReAct loop. `reflections` (default empty) is used only by Part B."""
    if max_steps is None:
        max_steps = MAX_STEPS

    trace = ""
    queries = []
    empty_searches = 0
    search_errors = 0
    invalid_actions = 0
    answer = None
    status = "step_limit"
    steps_used = 0

    reflection_block = ""
    if reflections and reflections.strip():
        reflection_block = ("\nReflections from earlier failed attempts "
                            "(use them to choose better, differently-worded search queries "
                            "and do not repeat failed queries):\n" + reflections + "\n")

    for step in range(1, max_steps + 1):
        steps_used = step
        prompt = (FEW_SHOT + reflection_block + "\nQuestion: " + question + "\n" + trace
                  + f"\nThought {step}:")

        output = chat(prompt).strip()
        action_type, parsed = parse_react_output(output)

        if action_type is None:
            invalid_actions += 1
            observation = ("Invalid action format. Return exactly one Thought and one Action. "
                           "Use SEARCH: or FINISH:.")
            trace += (f"\nThought {step}: Unable to parse model response.\n"
                      f"Action {step}: INVALID\n"
                      f"Observation {step}: {observation}\n")
            if verbose:
                print("\nRAW INVALID MODEL OUTPUT:")
                print(repr(output))
            continue

        action_input, thought = parsed

        if action_type == "FINISH":
            if not action_input:
                invalid_actions += 1
                trace += (f"\nThought {step}: {thought}\n"
                          f"Action {step}: INVALID\n"
                          f"Observation {step}: Empty final answer.\n")
                continue
            answer = action_input
            status = "finished"
            trace += f"\nThought {step}: {thought}\n" + f'Action {step}: FINISH: "{answer}"\n'
            break

        if action_type == "SEARCH":
            if not action_input:
                invalid_actions += 1
                trace += (f"\nThought {step}: {thought}\n"
                          f"Action {step}: INVALID\n"
                          f"Observation {step}: Empty search query.\n")
                continue
            query = action_input
            queries.append(query)
            observation, is_empty, is_error = web_search(query)   # global lookup (stress test swaps it)
            if is_empty:
                empty_searches += 1
            if is_error:
                search_errors += 1
            trace += (f"\nThought {step}: {thought}\n"
                      f'Action {step}: SEARCH: "{query}"\n'
                      f"Observation {step}: {observation}\n")

    normalized_queries = [" ".join(q.lower().split()) for q in queries]
    repeated_query = len(normalized_queries) != len(set(normalized_queries))

    if verbose:
        print("=" * 80)
        print("QUESTION:")
        print(question)
        print("=" * 80)
        print(trace)
        print("-" * 80)
        print("FINAL ANSWER:", answer)
        print("STATUS:", status)
        print("INVALID ACTIONS:", invalid_actions)
        print("=" * 80)

    return {
        "question": question, "answer": answer, "status": status, "steps": steps_used,
        "n_search": len(queries), "queries": queries, "empty_searches": empty_searches,
        "search_errors": search_errors, "invalid_actions": invalid_actions,
        "repeated_query": repeated_query, "trace": trace
    }


# ---------------------------------------------------------------- evaluation metrics (base paper)
def normalize_answer(text):
    text = (text or "").lower()
    text = "".join(char for char in text if char not in string.punctuation)
    text = re.sub(r"\b(a|an|the)\b", " ", text)
    return " ".join(text.split())


def exact_match(prediction, gold):
    return int(normalize_answer(prediction) == normalize_answer(gold))


def token_f1(prediction, gold):
    prediction_tokens = normalize_answer(prediction).split()
    gold_tokens = normalize_answer(gold).split()
    if not prediction_tokens or not gold_tokens:
        return float(prediction_tokens == gold_tokens)
    common = Counter(prediction_tokens) & Counter(gold_tokens)
    common_count = sum(common.values())
    if common_count == 0:
        return 0.0
    precision = common_count / len(prediction_tokens)
    recall = common_count / len(gold_tokens)
    return 2 * precision * recall / (precision + recall)


def score_result(result, gold):
    em = exact_match(result["answer"], gold)
    f1 = token_f1(result["answer"], gold)
    if result["status"] == "api_error":
        outcome = "API_ERROR"
    elif result["status"] != "finished":
        outcome = "NO_ANSWER (step limit)"
    elif em == 1:
        outcome = "EXACT_MATCH"
    elif f1 >= 0.5:
        outcome = "PARTIAL_MATCH"
    else:
        outcome = "WRONG"
    return em, f1, outcome


def _api_error_result(question, error):
    return {"question": question, "answer": None, "status": "api_error", "steps": 0, "n_search": 0,
            "queries": [], "empty_searches": 0, "search_errors": 0, "invalid_actions": 0,
            "repeated_query": False, "trace": f"API error: {error}",
            "attempts": 1, "first_attempt_failed": False, "recovered": False}


def _drop_failed_tail(results):
    for i, r in enumerate(results):
        if r.get("status") == "api_error":
            return results[:i]
    return results


def _valid_ckpt(results, rows):
    return all(i < len(rows) and r.get("question") == rows[i]["question"] for i, r in enumerate(results))


def run_batch(rows, ckpt_name=None):
    results = _drop_failed_tail(_ckpt_load(ckpt_name, []))
    if results and not _valid_ckpt(results, rows):
        print("Checkpoint does not match current sample; starting fresh.")
        results = []
    if results:
        print(f"Resuming from checkpoint: {len(results)}/{len(rows)} questions already done.")

    for i, row in enumerate(rows):
        if i < len(results):
            continue
        print(f"\nRunning Question {i+1}/{len(rows)}")
        try:
            result = react(row["question"])
        except RateLimitError:
            print("\nExperiment stopped due to the API rate limit. Partial results kept; rerun to resume.")
            break
        except Exception as error:
            print(f"API error on question {i+1}: {type(error).__name__}")
            result = _api_error_result(row["question"], error)

        result["gold"] = row["answer"]
        result["em"], result["f1"], result["outcome"] = score_result(result, row["answer"])
        results.append(result)
        _ckpt_save(ckpt_name, results)
        print(f"Q{i+1:02d} | {result['outcome']} | gold={row['answer']!r} | "
              f"got={result['answer']!r} | searches={result['n_search']}")
    return pd.DataFrame(results)


def show_summary(df, title="REACT BASELINE EVALUATION RESULTS"):
    if len(df) == 0:
        print("No questions completed (rate limit?). Nothing to summarise.")
        return
    total = len(df)
    print("=" * 55)
    print(title)
    print("=" * 55)
    print("Total Questions:", total)
    print("Exact Match:", f"{df['em'].mean():.1%}")
    print("Average Token F1:", f"{df['f1'].mean():.3f}")
    print("Completion Rate:", f"{(df['status'] == 'finished').mean():.1%}")
    print("Questions with Empty Search:", f"{(df['empty_searches'] > 0).mean():.1%}")
    print("Questions with Search Errors:", f"{(df['search_errors'] > 0).mean():.1%}")
    print("Repeated Query Frequency:", f"{df['repeated_query'].mean():.1%}")
    print("Average Searches per Question:", f"{df['n_search'].mean():.2f}")
    print("\nOutcome Distribution:")
    print(df["outcome"].value_counts().to_string())
    print("=" * 55)


def show_failures(df, k=3):
    if len(df) == 0:
        print("No results to show.")
        return
    failures = df[df["outcome"].isin(["WRONG", "NO_ANSWER (step limit)"])]
    print("Representative unsuccessful examples")
    for idx, row in failures.head(k).iterrows():
        print("\n" + "=" * 80)
        print(f"QUESTION {idx + 1}:")
        print(row["question"])
        print("\nGOLD ANSWER:")
        print(row["gold"])
        print("\nPREDICTED ANSWER:")
        print(row["answer"] if isinstance(row["answer"], str) else "(no answer)")
        print("\nOUTCOME:")
        print(row["outcome"])
        print("\nREACT TRACE:")
        print(row["trace"])


def load_sample():
    dataset = load_dataset("hotpotqa/hotpot_qa", "distractor", split="validation")
    dataset = dataset.select_columns(["question", "answer", "type", "level"])
    rows = [{"question": it["question"], "answer": it["answer"], "type": it["type"], "level": it["level"]}
            for it in dataset]
    hard_questions = [r for r in rows if r["type"] == "bridge" and r["level"] == "hard"]
    random.Random(SEED).shuffle(hard_questions)
    sample = hard_questions[:N_QUESTIONS]
    print("Total validation questions:", len(rows))
    print("Hard bridge questions:", len(hard_questions))
    print("Selected questions:", len(sample))
    print("\nSample question:")
    print(sample[0]["question"])
    return sample


def run_stress_test(sample):
    """Base-paper Section C: 30% of searches replaced by 'No results found.'"""
    global web_search
    print(f"Running Stress Test with {int(STRESS_RATE * 100)}% broken searches...\n")
    rng = random.Random(STRESS_SEED) if STRESS_SEED is not None else random

    def broken_web_search(query):
        if rng.random() < STRESS_RATE:
            return "No results found.", True, False
        return _original_web_search(query)

    web_search = broken_web_search
    try:
        df_stress = run_batch(sample, "stress")
    finally:
        web_search = _original_web_search
    if len(df_stress) == len(sample):
        print("\n--- STRESS TEST SUMMARY ---")
        show_summary(df_stress)
        print("\n--- STRESS TEST FAILURES ---")
        show_failures(df_stress, k=3)
    return df_stress


# =====================================================================================
#                 PART B : PROPOSED METHOD  (ReAct + Reflexion)
# =====================================================================================
def llm_call(system, user, max_tokens=800):
    return _call_llm([{"role": "system", "content": system}, {"role": "user", "content": user}],
                     max_tokens=max_tokens)


def generate_reflection(question, res, reason):
    prompt = (
        "A search-based question answering attempt failed. Do not answer the question.\n\n"
        f"Question: {question}\n"
        f"Search queries used: {res['queries']}\n"
        f"Failure reason: {reason}\n"
        f"Attempt trace (end):\n{res['trace'][-2500:]}\n\n"
        "Write exactly three short lines:\n"
        "Failure cause: <why it failed>\n"
        "Avoid: <strategy not to repeat>\n"
        "Next search query: <a better, differently-worded plain search query>\n"
        "No Markdown, no extra text.")
    text = llm_call("You are a reflection generator. You have no tools.", prompt)
    text = re.sub(r"<think>.*?</think>", "", text, flags=re.S).strip()
    return " | ".join(line.strip() for line in text.splitlines() if line.strip())


def attempt_ok(res):
    """Label-free operational evaluator: answer produced AND at least one non-empty observation."""
    evidence = res["n_search"] - res["empty_searches"] - res["search_errors"]
    if res["status"] != "finished" or not (res["answer"] or "").strip():
        return False, (f"No final answer within the step limit (empty searches={res['empty_searches']}, "
                       f"search errors={res['search_errors']}, invalid actions={res['invalid_actions']}).")
    if evidence <= 0:
        return False, "Answer produced without any non-empty search evidence."
    return True, "ok"


def react_reflexion(question, max_attempts=None):
    max_attempts = max_attempts or MAX_REFLEXION_ATTEMPTS
    memory, attempts = [], []
    for a in range(1, max_attempts + 1):
        refl = "\n".join(f"- {m}" for m in memory[-MAX_MEMORY_REFLECTIONS:])
        res = react(question, reflections=refl)
        ok, reason = attempt_ok(res)
        res["_ok"], res["_reason"], res["_refl_supplied"] = ok, reason, bool(refl)
        attempts.append(res)
        if ok:
            break
        if a < max_attempts:
            try:
                memory.append(generate_reflection(question, res, reason))
            except RateLimitError:
                raise
            except Exception as e:
                memory.append(f"(reflection failed: {type(e).__name__})")

    final = attempts[-1]
    queries = [q for r in attempts for q in r["queries"]]
    norm = [" ".join(q.lower().split()) for q in queries]
    first_failed = not attempts[0]["_ok"]
    return {
        "question": question,
        "answer": final["answer"] if final["_ok"] else None,
        "status": "finished" if final["_ok"] else final["status"],
        "steps": sum(r["steps"] for r in attempts),
        "n_search": len(queries), "queries": queries,
        "empty_searches": sum(r["empty_searches"] for r in attempts),
        "search_errors": sum(r["search_errors"] for r in attempts),
        "invalid_actions": sum(r["invalid_actions"] for r in attempts),
        "repeated_query": len(norm) != len(set(norm)),
        "attempts": len(attempts),
        "first_attempt_failed": first_failed,
        "recovered": bool(first_failed and final["_ok"]),
        "reflections": memory,
        "trace": "\n\n".join(f"##### ATTEMPT {i+1} ({'PASS' if r['_ok'] else 'FAIL: ' + r['_reason']}) #####\n"
                             + r["trace"] for i, r in enumerate(attempts)),
    }


def _faulty_search(rate, key):
    state = {"n": 0}

    def fn(query):
        state["n"] += 1
        if rate > 0 and random.Random(f"{key}-{state['n']}").random() < rate:
            return "No results found.", True, False
        return _original_web_search(query)
    return fn


def run_pair_batch(sample, rate, label):
    """Standard ReAct vs ReAct+Reflexion on the same questions with the SAME fault schedule.
    Returns (df_std, df_ref, complete)."""
    global web_search
    name = f"pair_{label}"
    ck = _ckpt_load(name, {"std": [], "ref": []})
    std_rows, ref_rows = ck["std"], ck["ref"]
    if len(std_rows) != len(ref_rows) or not _valid_ckpt(std_rows, sample):
        std_rows, ref_rows = [], []
    for i in range(len(std_rows)):
        if std_rows[i].get("status") == "api_error" or ref_rows[i].get("status") == "api_error":
            std_rows, ref_rows = std_rows[:i], ref_rows[:i]
            break
    if std_rows:
        print(f"[{label}] Resuming from checkpoint: {len(std_rows)}/{len(sample)} questions done.")

    complete = True
    for i, row in enumerate(sample):
        if i < len(std_rows):
            continue
        key = f"{FAULT_SEED}-{label}-{i}"
        print(f"\n[{label}] Question {i+1}/{len(sample)}")
        pair = []
        try:
            for arm, fn in (("Standard ", react), ("Reflexion", react_reflexion)):
                web_search = _faulty_search(rate, key)
                try:
                    res = fn(row["question"])
                except RateLimitError:
                    raise
                except Exception as e:
                    res = _api_error_result(row["question"], e)
                finally:
                    web_search = _original_web_search
                res["gold"] = row["answer"]
                res["em"], res["f1"], res["outcome"] = score_result(res, row["answer"])
                pair.append(res)
                extra = f" attempts={res.get('attempts', 1)}" if arm == "Reflexion" else ""
                print(f"  {arm}: {res['outcome']:<24} got={res['answer']!r} gold={row['answer']!r}{extra}")
        except RateLimitError:
            print("  Rate limit: stopping; progress saved. Rerun later to resume.")
            complete = False
            break
        std_rows.append(pair[0])
        ref_rows.append(pair[1])
        _ckpt_save(name, {"std": std_rows, "ref": ref_rows})
    return pd.DataFrame(std_rows), pd.DataFrame(ref_rows), complete


def print_pair_report(df_std, df_ref, label):
    show_summary(df_std, f"[{label}] STANDARD REACT (BASE PAPER)")
    show_summary(df_ref, f"[{label}] REACT + REFLEXION (PROPOSED)")
    if len(df_ref) == 0 or "first_attempt_failed" not in df_ref.columns:
        return
    ff = int(df_ref["first_attempt_failed"].sum())
    print(f"Reflexion: first attempt failed on {ff}/{len(df_ref)} questions; "
          f"recovered after reflection on {int(df_ref['recovered'].sum())}; "
          f"avg attempts = {df_ref['attempts'].mean():.2f}")
    wins = int((df_ref["em"] > df_std["em"]).sum())
    losses = int((df_ref["em"] < df_std["em"]).sum())
    print(f"Paired (EM): Reflexion-only correct={wins}, Standard-only correct={losses}, "
          f"delta EM={df_ref['em'].mean() - df_std['em'].mean():+.3f}, "
          f"delta F1={df_ref['f1'].mean() - df_std['f1'].mean():+.3f}")


# =====================================================================================
#                                      MAIN
# =====================================================================================
BASE_PAPER_REFERENCE = {
    "clean  (5 q)": {"EM": "20.0%", "F1": "0.467", "Completion": "60.0%", "Avg searches": "3.20",
                     "Empty/Errors/Repeated": "0% / 0% / 0%"},
    "stress (5 q)": {"EM": "20.0%", "F1": "0.333", "Completion": "40.0%", "Avg searches": "4.20",
                     "Empty/Errors/Repeated": "60% / 20% / 20%"},
}


def _incomplete(stage):
    print(f"\n*** {stage} is incomplete (rate limit). Progress is saved in ./{CKPT_DIR}. "
          f"Rerun this cell later (or after the quota resets) to continue. ***")
    print(f"Tokens used this run: {TOKENS_USED:,}")


def main():
    init_clients()

    demo_flag = os.path.join(CKPT_DIR, "_demo_done.txt")
    if RUN_DEMO and not os.path.exists(demo_flag):
        print("\n\n######## LIVE DEMONSTRATION (not part of benchmark) ########")
        demo_question = "Which university did the founder of the company that makes the Kindle attend?"
        try:
            react(demo_question, verbose=True)
        except RateLimitError:
            _incomplete("Demo")
            return
        with open(demo_flag, "w", encoding="utf-8") as f:
            f.write("done")

    print("\n\n######## HOTPOTQA DATASET ########")
    sample = load_sample()

    print("\n\n######## BASELINE RUN ########")
    df_baseline = run_batch(sample, "baseline")
    if len(df_baseline) < len(sample):
        _incomplete("Baseline run")
        return
    if (df_baseline["status"] == "api_error").any():
        print("\n*** Some questions ended in API_ERROR (see the error line printed above). "
              "Fix the cause and rerun; failed rows are not reused from checkpoints. ***")
        return
    show_summary(df_baseline)
    show_failures(df_baseline, k=3)

    print("\n\n######## SECTION C: 30% STRESS TEST ########")
    df_stress = run_stress_test(sample)
    if len(df_stress) < len(sample):
        _incomplete("Stress test")
        return

    df_baseline.to_csv("react_baseline_results.csv", index=False)
    pd.DataFrame(sample).to_csv("react_baseline_sample.csv", index=False)
    df_stress.to_csv("react_stress_results.csv", index=False)
    print("Experiment files saved successfully.")

    print("\n\n######## BASE-PAPER REPORTED VALUES (for comparison with the run above) ########")
    print(pd.DataFrame(BASE_PAPER_REFERENCE).to_string())

    if RUN_PROPOSED:
        print("\n\n######## PART B: PROPOSED METHOD (ReAct + Reflexion) ########")
        out = {}
        for label, rate in (("clean", 0.0), ("stress_30pct", STRESS_RATE)):
            d_std, d_ref, complete = run_pair_batch(sample, rate, label)
            print_pair_report(d_std, d_ref, label)
            d_std.to_csv(f"proposed_{label}_standard.csv", index=False)
            d_ref.to_csv(f"proposed_{label}_reflexion.csv", index=False)
            out[label] = (d_std, d_ref)
            if not complete:
                _incomplete(f"Part B ({label})")
                break
        if "stress_30pct" in out and len(out["stress_30pct"][1]) == len(sample):
            print("\nFailure traces (Reflexion, stress condition):")
            show_failures(out["stress_30pct"][1], k=2)

    print(f"\nTotal tokens used this run: {TOKENS_USED:,}")

    try:
        from google.colab import files
        for f in os.listdir("."):
            if f.endswith(".csv") and (f.startswith("react_") or f.startswith("proposed_")):
                files.download(f)
    except Exception:
        pass


if __name__ == "__main__":
    main()

Gemini and Tavily connections initialized successfully.
Available Gemini models: ['gemini-2.5-flash', 'gemini-2.5-pro', 'gemini-2.5-flash-preview-tts', 'gemini-2.5-pro-preview-tts', 'gemini-flash-latest', 'gemini-flash-lite-latest', 'gemini-pro-latest', 'gemini-2.5-flash-lite', 'gemini-2.5-flash-image', 'gemini-3-flash-preview', 'gemini-3.1-pro-preview', 'gemini-3.1-pro-preview-customtools', 'gemini-3.1-flash-lite-preview', 'gemini-3.1-flash-lite', 'gemini-3-pro-image-preview', 'gemini-3-pro-image', 'gemini-3.1-flash-image-preview', 'gemini-3.1-flash-image', 'gemini-3.1-flash-lite-image', 'gemini-nano-banana-2.1', 'gemini-3.5-flash', 'gemini-3.5-flash-lite', 'gemini-omni-flash-preview', 'gemini-omni-1.1-flash', 'gemini-omni-1.2-flash', 'gemini-3.5-transcribe', 'gemini-3.6-flash', 'gemini-3.7-flash', 'gemini-3.8-flash', 'gemini-4-argon', 'gemini-3.1-flash-tts-preview', 'gemini-3.8-flash-tts', 'gemini-3.8-flash-lite-tts', 'gemini-robotics-er-2-preview', 'gemini-2.5-computer-use-preview-1

README.md:   0%|          | 0.00/9.52k [00:00<?, ?B/s]

distractor/train-00000-of-00002.parquet: reconstructing file:   0%|          |  0.00B /  166MB            

distractor/train-00000-of-00002.parquet: downloading bytes:           |  0.00B            

distractor/train-00001-of-00002.parquet: reconstructing file:   0%|          |  0.00B /  166MB            

distractor/train-00001-of-00002.parquet: downloading bytes:           |  0.00B            

... (15 outputs left out)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>